In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, FunctionTransformer, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [3]:
train = pd.read_csv(r'C:\Users\Abhay\Downloads\Train.csv')

In [4]:
test = pd.read_csv(r'C:\Users\Abhay\Downloads\Test.csv')

In [5]:
train.sample(5)

,Unnamed: 0,url,Phish?
2717559,2717559,abigchocolateslut.blogspot.qa,1
629329,629329,ancient-technology.com/ancient_technology_es/s...,0
885336,885336,english.manartv.com.lb/366617,0
864904,864904,twinkletells.blogspot.ca,1
2402776,2402776,www.geostab.cz/,0


In [6]:
test = test.fillna(0)

In [7]:
### Removing the Unnamed Column from both the dataset Train and Test
train = train.drop('Unnamed: 0', axis = 1)
test = test.drop('Unnamed: 0', axis = 1)

In [8]:
train.sample(5)

,url,Phish?
637368,www.travelnews.tw/news/premia-partners%e7%99%b...,0
3456959,commondreams.org/news2003/0926-01.htm,1
3091424,epaper.cqrb.cn/html/kjb/2018-07/10/07/content_...,0
2987215,big-boobs-sex.blogspot.ba,1
3559467,webcam-hoeren.startspot.nl,1


In [9]:
train.isnull().sum()

url       1
Phish?    0
dtype: int64

In [10]:
test.isnull().sum()

url       0
Phish?    0
dtype: int64

In [16]:
df_taken_1 = train.sample(25000)
df_taken_1

,url,Phish?
2887423,www.lkz.de/sport-uebersicht/sport-ueberregiona...,0
2213518,b4max.blogspot.gr,1
3587648,ningde.gov.cn/hdjl/zxft/zqyg/201805/t20180515_...,0
3888843,babestvgirl.blogspot.fi,1
619054,x35.ssftkdfm.com,1
...,...,...
2215675,a-really-trill-motherfucker.tumblr.com,1
2756351,www.insee.fr/fr/statistiques/2043858?geo=epci-...,0
1651614,dryermaster.com/privacy,0
276054,natalieslingerie.blogspot.be,1


In [19]:
df_taken_2 = test.sample(25000)
df_taken_2['Phish?'] = df_taken_2['Phish?'].astype(int)
df_taken_2

,url,Phish?
1104606,giovanebisex.blogspot.be,1
1610075,traffic.tiles.virtualearth.net,1
300379,oshpd.ca.gov,1
286299,gsp-aeu001-se06.gamesparks.net,1
1537268,www.therabreath.com/sweepstakes/,0
...,...,...
883327,community.fitforfun.de/habt-ihr-schon-mal-frem...,0
2464928,groverpro.com/will-james-signature-snare-drum-...,0
2187169,www.faune-aquitaine.org/index.php?m_id=54&mid=...,0
1040092,hotels4teams.com,1


In [23]:
main_df = pd.concat([df_taken_1, df_taken_2], axis = 0)

In [43]:
df = main_df.reset_index(drop = True)

In [44]:
df.head(5)

,url,Phish?
0,www.lkz.de/sport-uebersicht/sport-ueberregiona...,0
1,b4max.blogspot.gr,1
2,ningde.gov.cn/hdjl/zxft/zqyg/201805/t20180515_...,0
3,babestvgirl.blogspot.fi,1
4,x35.ssftkdfm.com,1


In [45]:
df.shape

(50000, 2)

In [46]:
df.to_csv('Dataset.csv')

In [47]:
import re
import tldextract
import math
from urllib.parse import urlparse

def shannon_entropy(string):
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    entropy = -sum([p * math.log2(p) for p in prob])
    return entropy

def has_ip(domain):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, domain) else 0

def count_digits(s):
    return sum(c.isdigit() for c in s)

def count_letters(s):
    return sum(c.isalpha() for c in s)

def count_special_chars(s):
    return len(re.findall(r'[^a-zA-Z0-9]', s))

def count_words(s):
    words = re.split(r'[\W_]+', s)
    words = [w for w in words if w]
    return len(words)

suspicious_keywords = [
    'login', 'secure', 'update', 'bank', 'account',
    'verify', 'paypal', 'signin', 'confirm', 'free',
    'webscr', 'ebay', 'amazon'
]

def keyword_count(url):
    return sum(word in url.lower() for word in suspicious_keywords)

def extract_features(url):

    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    domain = ext.domain
    subdomain = ext.subdomain
    path = parsed.path
    
    features = {}
    
    # Basic Length Features
    features['url_length'] = len(url)
    features['domain_length'] = len(domain)
    features['subdomain_length'] = len(subdomain)
    features['path_length'] = len(path)
    
    # Count Features
    features['._count'] = url.count('.')
    features['-_count'] = url.count('-')
    features['__count'] = url.count('_')
    features['/_count'] = url.count('/')
    features['?_count'] = url.count('?')
    features['=_count'] = url.count('=')
    features['@_count'] = url.count('@')
    
    # Digit / Letter Features
    features['digit_count'] = count_digits(url)
    features['letter_count'] = count_letters(url)
    features['special_char_count'] = count_special_chars(url)
    
    # Ratio Features
    features['digit_ratio'] = features['digit_count'] / len(url)
    features['letter_ratio'] = features['letter_count'] / len(url)
    
    # Domain-based
    features['has_ip'] = has_ip(url)
    features['entropy'] = shannon_entropy(url)
    
    # Word-based
    features['word_count'] = count_words(url)
    features['keyword_count'] = keyword_count(url)
    
    # TLD suspicious
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'gq']
    features['suspicious_tld'] = 1 if ext.suffix in suspicious_tlds else 0
    
    # HTTPS
    features['https'] = 1 if parsed.scheme == 'https' else 0
    
    return features

def build_feature_dataframe(df, url_column):
    feature_list = df[url_column].apply(lambda x: extract_features(x))
    feature_df = pd.DataFrame(feature_list.tolist())
    return feature_df

In [48]:
data = build_feature_dataframe(df, 'url')

In [49]:
data.head(5)

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,letter_count,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,https
0,139,3,3,139,3,10,2,3,0,0,...,109,22,0.057554,0.784173,0,4.707580,21,0,0,0
1,17,8,5,17,2,0,0,0,0,0,...,14,2,0.058824,0.823529,0,3.616875,3,0,0,0
2,56,6,0,56,3,0,1,5,0,0,...,27,9,0.357143,0.482143,0,4.662016,10,0,0,0
3,23,8,11,23,2,0,0,0,0,0,...,21,2,0.000000,0.913043,0,3.708132,3,0,0,0
4,16,8,3,16,2,0,0,0,0,0,...,12,2,0.125000,0.750000,0,3.500000,3,0,0,0


In [50]:
data['Target'] = df['Phish?']

In [51]:
data

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,https,Target
0,139,3,3,139,3,10,2,3,0,0,...,22,0.057554,0.784173,0,4.707580,21,0,0,0,0
1,17,8,5,17,2,0,0,0,0,0,...,2,0.058824,0.823529,0,3.616875,3,0,0,0,1
2,56,6,0,56,3,0,1,5,0,0,...,9,0.357143,0.482143,0,4.662016,10,0,0,0,0
3,23,8,11,23,2,0,0,0,0,0,...,2,0.000000,0.913043,0,3.708132,3,0,0,0,1
4,16,8,3,16,2,0,0,0,0,0,...,2,0.125000,0.750000,0,3.500000,3,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,140,9,9,140,2,12,0,3,0,0,...,17,0.142857,0.735714,0,4.694287,18,0,0,0,0
49996,51,9,0,51,1,5,0,1,0,0,...,7,0.000000,0.862745,0,4.147943,8,0,0,0,0
49997,61,15,3,33,3,1,1,1,1,3,...,12,0.131148,0.672131,0,4.754531,13,0,0,0,0
49998,16,12,0,16,1,0,0,0,0,0,...,1,0.062500,0.875000,0,3.375000,2,0,0,0,1


In [52]:
data.to_csv('Dataset.csv')

In [58]:
data.isnull().sum()

url_length            0
domain_length         0
subdomain_length      0
path_length           0
._count               0
-_count               0
__count               0
/_count               0
?_count               0
=_count               0
@_count               0
digit_count           0
letter_count          0
special_char_count    0
digit_ratio           0
letter_ratio          0
has_ip                0
entropy               0
word_count            0
keyword_count         0
suspicious_tld        0
https                 0
Target                0
dtype: int64

In [60]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   url_length          50000 non-null  int64  
 1   domain_length       50000 non-null  int64  
 2   subdomain_length    50000 non-null  int64  
 3   path_length         50000 non-null  int64  
 4   ._count             50000 non-null  int64  
 5   -_count             50000 non-null  int64  
 6   __count             50000 non-null  int64  
 7   /_count             50000 non-null  int64  
 8   ?_count             50000 non-null  int64  
 9   =_count             50000 non-null  int64  
 10  @_count             50000 non-null  int64  
 11  digit_count         50000 non-null  int64  
 12  letter_count        50000 non-null  int64  
 13  special_char_count  50000 non-null  int64  
 14  digit_ratio         50000 non-null  float64
 15  letter_ratio        50000 non-null  float64
 16  has_

In [61]:
! git status

On branch Abhay
Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Data_Fetching.ipynb
	deleted:    .ipynb_checkpoints/Feature_Engineering-checkpoint.ipynb
	deleted:    Feature_Engineering.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.ipynb_checkpoints/
	.ipynb_checkpoints/Data_Preprocessing-checkpoint.ipynb
	.ipynb_checkpoints/Dataset-checkpoint.csv
	Data_Preprocessing.ipynb
	Dataset.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [62]:
! git add .

In [63]:
! git status

On branch Abhay
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	renamed:    .ipynb_checkpoints/Feature_Engineering-checkpoint.ipynb -> .ipynb_checkpoints/Data_Preprocessing-checkpoint.ipynb
	new file:   .ipynb_checkpoints/Dataset-checkpoint.csv
	new file:   Data_Preprocessing.ipynb
	new file:   Dataset.csv
	deleted:    Feature_Engineering.ipynb

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Data_Fetching.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.ipynb_checkpoints/



In [64]:
! git commit -m 'Data_preprocessing'

[Abhay 59a4644] 'Data_preprocessing'
 5 files changed, 101505 insertions(+), 480 deletions(-)
 rename ML/.ipynb_checkpoints/{Feature_Engineering-checkpoint.ipynb => Data_Preprocessing-checkpoint.ipynb} (100%)
 create mode 100644 ML/.ipynb_checkpoints/Dataset-checkpoint.csv
 create mode 100644 ML/Data_Preprocessing.ipynb
 create mode 100644 ML/Dataset.csv
 delete mode 100644 ML/Feature_Engineering.ipynb


In [65]:
! git push origin Abhay

To https://github.com/MokshJn/Phish-Secure
   a86d777..59a4644  Abhay -> Abhay


In [1]:
! git branch

* Abhay
  main


In [2]:
! git checkout main

error: Your local changes to the following files would be overwritten by checkout:
	ML/Final_model.ipynb
Please commit your changes or stash them before you switch branches.
Aborting


In [3]:
! git branch

* Abhay
  main
